# Playground Series S6E7 — Predicting Student Health Risk
## Score: .94932

In [29]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")

SEED = 42
N_SPLITS = 5
DATA_DIR = Path("playground-series-s6e7")
TARGET = "health_condition"
ID_COL = "id"

rng = np.random.default_rng(SEED)
print("LightGBM", lgb.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

LightGBM 4.6.0 | pandas 3.0.3 | numpy 2.4.6


In [30]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("train:", train.shape, "| test:", test.shape)

CAT_COLS = ["diet_type", "stress_level", "sleep_quality",
            "physical_activity_level", "smoking_alcohol", "gender"]
NUM_COLS = ["sleep_duration", "heart_rate", "bmi", "calorie_expenditure",
            "step_count", "exercise_duration", "water_intake"]

CLASSES = ["at-risk", "unhealthy", "fit"]
class_to_int = {c: i for i, c in enumerate(CLASSES)}
int_to_class = {i: c for c, i in class_to_int.items()}
y = train[TARGET].map(class_to_int).to_numpy()

print("\nTarget distribution:")
print(train[TARGET].value_counts(normalize=True).round(4))

train: (690088, 15) | test: (295753, 14)

Target distribution:
health_condition
at-risk      0.8587
unhealthy    0.0836
fit          0.0577
Name: proportion, dtype: float64


## Feature preparation

In [31]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    X = df.copy()

    X["n_missing"] = df[NUM_COLS + CAT_COLS].isna().sum(axis=1)

    X["steps_per_cal"] = X["step_count"] / X["calorie_expenditure"]
    X["cal_per_min_exercise"] = X["calorie_expenditure"] / (X["exercise_duration"] + 1e-3)
    X["steps_per_min_exercise"] = X["step_count"] / (X["exercise_duration"] + 1e-3)

    for c in CAT_COLS:
        X[c] = X[c].fillna("__NA__").astype("category")

    return X


FEATURES = NUM_COLS + CAT_COLS + [
    "n_missing", "steps_per_cal", "cal_per_min_exercise", "steps_per_min_exercise",
]

X = build_features(train)[FEATURES].copy()
X_test = build_features(test)[FEATURES].copy()

for c in CAT_COLS:
    levels = X[c].cat.categories.union(X_test[c].cat.categories)
    X[c] = X[c].cat.set_categories(levels)
    X_test[c] = X_test[c].cat.set_categories(levels)

# --- C: distributional transforms (percentile + z-score + per-row summaries) ---
_alln = pd.concat([train[NUM_COLS], test[NUM_COLS]], ignore_index=True)
mu = _alln.mean()
sd = _alln.std() + 1e-6

DIST_FEATS = []
ztr, zte = {}, {}
for n in NUM_COLS:
    ranks = _alln[n].rank(pct=True)
    X[f"{n}_pct"] = ranks.iloc[:len(train)].to_numpy()
    X_test[f"{n}_pct"] = ranks.iloc[len(train):].to_numpy()
    ztr[n] = (train[n].to_numpy() - mu[n]) / sd[n]
    zte[n] = (test[n].to_numpy() - mu[n]) / sd[n]
    X[f"{n}_z"] = ztr[n]
    X_test[f"{n}_z"] = zte[n]
    DIST_FEATS += [f"{n}_pct", f"{n}_z"]

for D, zd in [(X, ztr), (X_test, zte)]:
    Z = np.vstack([zd[n] for n in NUM_COLS]).T
    D["z_absmean"] = np.nanmean(np.abs(Z), axis=1)
    D["z_max"] = np.nanmax(Z, axis=1)
    D["z_min"] = np.nanmin(Z, axis=1)
    D["z_range"] = D["z_max"] - D["z_min"]
    D["z_nextreme"] = np.nansum(np.abs(Z) > 2, axis=1)
DIST_FEATS += ["z_absmean", "z_max", "z_min", "z_range", "z_nextreme"]

FEATURES = FEATURES + DIST_FEATS

# --- C: fine quantile bins of the top rule-driving numerics (categorical thresholds) ---
BIN_COLS = ["sleep_duration", "bmi", "exercise_duration", "step_count", "calorie_expenditure"]
N_BINS = 32
BIN_CATS = []
for n in BIN_COLS:
    edges = np.unique(np.nanquantile(_alln[n], np.linspace(0, 1, N_BINS + 1)))
    name = f"{n}_bin"
    X[name] = pd.cut(train[n], bins=edges, labels=False, include_lowest=True)
    X_test[name] = pd.cut(test[n], bins=edges, labels=False, include_lowest=True)
    X[name] = X[name].astype("Int64").astype("category")
    X_test[name] = X_test[name].astype("Int64").astype("category")
    BIN_CATS.append(name)

for c in BIN_CATS:
    levels = X[c].cat.categories.union(X_test[c].cat.categories)
    X[c] = X[c].cat.set_categories(levels)
    X_test[c] = X_test[c].cat.set_categories(levels)

CAT_FEATURES = list(CAT_COLS) + BIN_CATS
FEATURES = FEATURES + BIN_CATS
print("Features:", len(FEATURES), "| added distributional:", len(DIST_FEATS), "| bins:", len(BIN_CATS))
print(X.dtypes)

Features: 41 | added distributional: 19 | bins: 5
sleep_duration              float64
heart_rate                  float64
bmi                         float64
calorie_expenditure         float64
step_count                  float64
exercise_duration           float64
water_intake                float64
diet_type                  category
stress_level               category
sleep_quality              category
physical_activity_level    category
smoking_alcohol            category
gender                     category
n_missing                     int64
steps_per_cal               float64
cal_per_min_exercise        float64
steps_per_min_exercise      float64
sleep_duration_pct          float64
sleep_duration_z            float64
heart_rate_pct              float64
heart_rate_z                float64
bmi_pct                     float64
bmi_z                       float64
calorie_expenditure_pct     float64
calorie_expenditure_z       float64
step_count_pct              float64
step_count_z  

## Cross-validated training

In [32]:
lgb_params = dict(
    objective="multiclass",
    num_class=len(CLASSES),
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
)

PL_ROUNDS = 2
PL_THRESH = {0: 0.97, 1: 0.85, 2: 0.85}  # per-class: strict for at-risk, looser for minorities
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)


def run_cv(extra_X=None, extra_y=None):
    oof = np.zeros((len(X), len(CLASSES)))
    tst = np.zeros((len(X_test), len(CLASSES)))
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        if extra_X is None:
            Xtr, ytr = X.iloc[tr_idx], y[tr_idx]
        else:
            Xtr = pd.concat([X.iloc[tr_idx], extra_X], ignore_index=True)
            ytr = np.r_[y[tr_idx], extra_y]
        model = lgb.LGBMClassifier(**lgb_params)
        model.fit(
            Xtr, ytr,
            eval_set=[(X.iloc[va_idx], y[va_idx])],
            eval_metric="multi_logloss",
            categorical_feature=CAT_FEATURES,
            callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
        )
        oof[va_idx] = model.predict_proba(X.iloc[va_idx])
        tst += model.predict_proba(X_test) / N_SPLITS
    return oof, tst


# Round 0: baseline CV to get initial test probabilities
oof_proba, test_proba = run_cv()
print(f"round 0 OOF argmax bal-acc: {balanced_accuracy_score(y, oof_proba.argmax(1)):.5f}")

# Iterative pseudo-labeling with per-class confidence thresholds
# (test rows never enter OOF folds -> OOF stays leakage-free)
for r in range(1, PL_ROUNDS + 1):
    pred = test_proba.argmax(1)
    conf = test_proba.max(1)
    keep = np.zeros(len(X_test), dtype=bool)
    for c in range(len(CLASSES)):
        keep |= (pred == c) & (conf >= PL_THRESH[c])

    X_pl = X_test.iloc[np.flatnonzero(keep)].reset_index(drop=True)
    y_pl = pred[keep]
    counts = pd.Series(y_pl).map(int_to_class).value_counts().to_dict()

    oof_proba, test_proba = run_cv(X_pl, y_pl)
    bal = balanced_accuracy_score(y, oof_proba.argmax(1))
    print(f"round {r}: pseudo={keep.sum()} {counts} | OOF argmax bal-acc={bal:.5f}")

argmax_oof = oof_proba.argmax(axis=1)

Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's multi_logloss: 0.109229
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's multi_logloss: 0.110142
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's multi_logloss: 0.113212
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's multi_logloss: 0.109515
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's multi_logloss: 0.11276
round 0 OOF argmax bal-acc: 0.92988
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's multi_logloss: 0.11283
Training until validation scores don't improve for 100 round

## Tune the decision rule for balanced accuracy

In [33]:
from scipy.optimize import minimize
from sklearn.isotonic import IsotonicRegression


def weighted_predict(proba: np.ndarray, w: np.ndarray) -> np.ndarray:
    return (proba * w).argmax(axis=1)


def _neg_bal(w2, proba, y_true):
    w = np.array([1.0, w2[0], w2[1]])
    return -balanced_accuracy_score(y_true, weighted_predict(proba, w))


def optimize_weights(proba, y_true, grid=None):
    grid = np.geomspace(0.3, 30, 30) if grid is None else grid
    best_w = np.array([1.0, 1.0, 1.0])
    best = -_neg_bal([1.0, 1.0], proba, y_true)
    for wu in grid:
        for wf in grid:
            s = -_neg_bal([wu, wf], proba, y_true)
            if s > best:
                best, best_w = s, np.array([1.0, wu, wf])
    res = minimize(_neg_bal, x0=best_w[1:], args=(proba, y_true),
                   method="Nelder-Mead", options=dict(xatol=1e-3, fatol=1e-7, maxiter=1000))
    if -res.fun > best:
        best, best_w = -res.fun, np.array([1.0, res.x[0], res.x[1]])
    return best_w, best


def calibrate(oof_p, test_p, y_true):
    oc, tc = np.zeros_like(oof_p), np.zeros_like(test_p)
    for c in range(oof_p.shape[1]):
        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(oof_p[:, c], (y_true == c).astype(float))
        oc[:, c] = iso.transform(oof_p[:, c])
        tc[:, c] = iso.transform(test_p[:, c])
    oc /= oc.sum(axis=1, keepdims=True)
    tc /= tc.sum(axis=1, keepdims=True)
    return oc, tc


w_raw, s_raw = optimize_weights(oof_proba, y)
oof_cal, test_cal = calibrate(oof_proba, test_proba, y)
w_cal, s_cal = optimize_weights(oof_cal, y)

print(f"argmax (no tuning): {balanced_accuracy_score(y, argmax_oof):.5f}")
print(f"raw blend        : {s_raw:.5f}  w={w_raw.round(3)}")
print(f"calibrated blend : {s_cal:.5f}  w={w_cal.round(3)}")

if s_cal >= s_raw:
    oof_final, test_final, best_w, best_score = oof_cal, test_cal, w_cal, s_cal
    print("-> using CALIBRATED probabilities")
else:
    oof_final, test_final, best_w, best_score = oof_proba, test_proba, w_raw, s_raw
    print("-> using RAW probabilities")

tuned_oof = weighted_predict(oof_final, best_w)
print(f"\nFinal OOF balanced accuracy (tuned): {best_score:.5f}")
print("\nConfusion matrix (rows=true, cols=pred) [at-risk, unhealthy, fit]:")
print(confusion_matrix(y, tuned_oof))
print("\n", classification_report(y, tuned_oof, target_names=CLASSES, digits=4))

argmax (no tuning): 0.93446
raw blend        : 0.94905  w=[ 1.     7.137 13.656]
calibrated blend : 0.94919  w=[ 1.    10.048 17.578]
-> using CALIBRATED probabilities

Final OOF balanced accuracy (tuned): 0.94919

Confusion matrix (rows=true, cols=pred) [at-risk, unhealthy, fit]:
[[552245  26250  14066]
 [  1720  55759    245]
 [  1785    219  37799]]

               precision    recall  f1-score   support

     at-risk     0.9937    0.9320    0.9618    592561
   unhealthy     0.6781    0.9660    0.7968     57724
         fit     0.7254    0.9497    0.8225     39803

    accuracy                         0.9358    690088
   macro avg     0.7991    0.9492    0.8604    690088
weighted avg     0.9518    0.9358    0.9400    690088



## Predict test set & write submission

In [34]:
test_pred_int = weighted_predict(test_final, best_w)
test_pred_lbl = pd.Series(test_pred_int).map(int_to_class)

submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_pred_lbl})
submission.to_csv("submission.csv", index=False)

print("Wrote submission.csv:", submission.shape)
print("\nPredicted label distribution:")
print(submission[TARGET].value_counts(normalize=True).round(4))
print("\nTrain label distribution (for reference):")
print(train[TARGET].value_counts(normalize=True).round(4))
submission.head()

Wrote submission.csv: (295753, 2)

Predicted label distribution:
health_condition
at-risk      0.8061
unhealthy    0.1194
fit          0.0745
Name: proportion, dtype: float64

Train label distribution (for reference):
health_condition
at-risk      0.8587
unhealthy    0.0836
fit          0.0577
Name: proportion, dtype: float64


,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


## Diagnostics — is there headroom?

In [35]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score


def encode(df):
    e = df.copy()
    for c in CAT_COLS:
        e[c] = e[c].cat.codes
    return e.fillna(-999)


# 1) Adversarial validation: can a model tell train from test?
Xa = pd.concat([encode(X), encode(X_test)], ignore_index=True)
ya = np.r_[np.zeros(len(X)), np.ones(len(X_test))]
Xa_tr, Xa_va, ya_tr, ya_va = train_test_split(Xa, ya, test_size=0.3, stratify=ya, random_state=SEED)
adv = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                         random_state=SEED, n_jobs=-1, verbose=-1)
adv.fit(Xa_tr, ya_tr)
adv_auc = roc_auc_score(ya_va, adv.predict_proba(Xa_va)[:, 1])
print(f"[1] Adversarial AUC (0.50 = no train/test shift): {adv_auc:.4f}")

# 2) Mutual information of each feature with the target
Xe = encode(X)
cat_idx = [Xe.columns.get_loc(c) for c in CAT_COLS]
mi = mutual_info_classif(Xe, y, discrete_features=cat_idx, random_state=SEED)
print("\n[2] Mutual information with target (higher = more signal):")
print(pd.Series(mi, index=Xe.columns).sort_values(ascending=False).round(4).to_string())

# 3) High-capacity ceiling probe (deep LGBM on a holdout)
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
probe = lgb.LGBMClassifier(objective="multiclass", num_class=len(CLASSES),
    n_estimators=3000, learning_rate=0.05, num_leaves=255, min_child_samples=50,
    class_weight="balanced", random_state=SEED, n_jobs=-1, verbose=-1)
probe.fit(Xtr, ytr, categorical_feature=CAT_COLS)
print(f"\n[3] High-capacity probe val balanced-acc (argmax): "
      f"{balanced_accuracy_score(yva, probe.predict(Xva)):.5f}")

# 4) Is the target rule-like? Decision trees at increasing depth
Xte = encode(X)
Xtr, Xva, ytr, yva = train_test_split(Xte, y, test_size=0.2, stratify=y, random_state=SEED)
print("\n[4] Decision-tree depth sweep (a big jump = rule-based target):")
for d in [3, 5, 8, 12, None]:
    t = DecisionTreeClassifier(max_depth=d, class_weight="balanced", random_state=SEED)
    t.fit(Xtr, ytr)
    print(f"    depth={str(d):>4}: val bal-acc={balanced_accuracy_score(yva, t.predict(Xva)):.5f}")

TypeError: Cannot setitem on a Categorical with a new category (-999), set the categories first